In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'collision-walking-rand'  # ckpt = 4000
# exp_name = 'friction-walking-fractal-kp2000kd50'
# exp_name = 'friction-walking-terrain2-kp2000kd50-relinvel20'  # ckpt = 20000
exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-6'  # ckpt = 20000
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-6'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-5'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.001
# env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 100.0]}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.0172, -0.4736, -0.7030,  0.5600, -1.1907,  0.1431, -0.1918,  0.5651,
          0.6153, -0.0524, -1.3241,  0.2163]], device='cuda:0')
Scaled actions :  tensor([[-0.0172, -0.4736, -0.7030,  0.5600, -1.1907,  0.1431, -0.1918,  0.5651,
          0.6153, -0.0524, -1.3241,  0.2163]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0711e-10,  2.0808e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.4362e-08,
          3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04, -3.4389e-08,
         -8.5621e-08, -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,
          7.6633e-07,  7.1809e-07,  1.5422e-06, -7.0859e-03,  1.8227e-02,
         -9.5517e-03, -1.7194e-06, -4.2811e-06, -2.3818e-06, -7.0895e-03,
          1.8232e-02, -9.5524e-03,  3.8316e-05, -1.7207e-02, -4.7359e-01,
         -7.0297e-01,  5.6004e-01, -1.1907e+00,  1.4310e-01, -1.9184e-01,
          5.6513e-01,  6.1531e-01, -5.2368e-02, -1.3241e+00,  2.1630e-01]],
       device='cuda:0')
torques: [ 3.99683909e-16 -9.12643102e-16  2.42782550e-06  7.51007967e-06
  1.66224901e-06  2.18425700e-16  2.17253195e-16 -6.61016595e-16
  2.42782550e-06  7.51007967e-06  1.66224901e-06 -7.93165041e-17]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-8.6644e-04, -6.0891e-01, -9.5810e-01,  2.4884e-01, -1.1872e+00,
          4.4494e-02, -1.0160e-01,  5.8894e-01,  9.3716e-01,  6.4856e-01,
         -1.4796e+00,  3.9429e-01]], device='cuda:0')
Scaled actions :  tensor([[-8.6644e-04, -6.0891e-01, -9.5810e-01,  2.4884e-01, -1.1872e+00,
          4.4494e-02, -1.0160e-01,  5.8894e-01,  9.3716e-01,  6.4856e-01,
         -1.4796e+00,  3.9429e-01]], device='cuda:0')
obs :  tensor([[-1.9230e-01, -1.1243e-01,  6.5862e-01, -2.8022e-03,  4.0301e-03,
         -9.9999e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -5.7831e-03,
         -3.8981e-03, -1.9766e-02,  3.8223e-02, -1.0789e-01,  5.1091e-02,
         -3.5310e-02, -1.8959e-03,  6.5516e-03,  1.8113e-02, -9.0899e-02,
          4.1945e-02, -2.9252e-02, -2.9009e-02, -1.7038e-01,  3.2490e-01,
         -9.7179e-01,  2.3425e-01, -2.6394e-01, -4.3161e-03,  8.1138e-02,
          1.2250e-01, -8.1053e-01,  2.9552e-01, -8.6644e-04, -6.0891e-01,
         -9.5810e-01,  2.

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[-0.0827, -0.4567, -0.0780, -0.4868,  0.2576, -0.2334,  0.4965, -0.5503,
          0.1251,  0.0363, -0.4306,  0.2787]], device='cuda:0')
Scaled actions :  tensor([[-0.0827, -0.4567, -0.0780, -0.4868,  0.2576, -0.2334,  0.4965, -0.5503,
          0.1251,  0.0363, -0.4306,  0.2787]], device='cuda:0')
obs :  tensor([[-2.3830e-01, -1.2359e-01,  7.7055e-01, -7.5848e-03,  1.3125e-02,
         -9.9989e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -4.6759e-03,
         -1.5933e-02, -6.5383e-02,  1.1801e-01, -3.7234e-01,  5.4766e-02,
         -6.8739e-02,  1.6705e-03,  2.5553e-02,  6.0611e-02, -3.5982e-01,
          1.4620e-01,  1.4434e-02, -9.2759e-02, -2.6686e-01,  4.2468e-01,
         -1.5372e+00, -1.5796e-02, -1.0836e-01,  4.5119e-02,  1.1162e-01,
          2.6564e-01, -1.7882e+00,  5.3343e-01, -8.2683e-02, -4.5672e-01,
         -7.8018e-02, -4.8679e-01,  2.5759e-01, -2.3341e-01,  4.9647e-01,
         -5.5028e-01,  1.2511e-01,  3.6299e-02, -4.3065e-01,  2.7

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[-0.3079,  0.0461,  0.1867, -0.6947,  0.6413,  0.1835,  0.4520,  0.2705,
          0.2744, -0.0351, -0.0712,  0.4467]], device='cuda:0')
Scaled actions :  tensor([[-0.3079,  0.0461,  0.1867, -0.6947,  0.6413,  0.1835,  0.4520,  0.2705,
          0.2744, -0.0351, -0.0712,  0.4467]], device='cuda:0')
obs :  tensor([[ 0.3731, -0.2778,  0.4971, -0.0165,  0.0097, -0.9998,  1.0000,  0.0000,
          0.0000, -0.0218, -0.0599, -0.0946,  0.1672, -0.5740, -0.0228, -0.0361,
         -0.0164,  0.0630,  0.0902, -0.6412,  0.2070, -0.1001, -0.3358, -0.0505,
          0.0947, -0.5778, -0.4529,  0.3923, -0.1955,  0.2202,  0.0639, -0.9877,
          0.1676, -0.3079,  0.0461,  0.1867, -0.6947,  0.6413,  0.1835,  0.4520,
          0.2705,  0.2744, -0.0351, -0.0712,  0.4467]], device='cuda:0')
torques: [ -24.35986693 -200.           97.23663126 -200.          200.
   29.83075919  200.         -200.          -84.85544412 -184.5279308
  200.          -27.47982509]
データ収集: 

In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[ 0.1443,  1.2860, -1.1694, -0.3019, -0.7479,  0.3948, -0.1507,  0.4846,
         -0.3260, -0.4666, -0.5843,  0.3834]], device='cuda:0')
Scaled actions :  tensor([[ 0.1443,  1.2860, -1.1694, -0.3019, -0.7479,  0.3948, -0.1507,  0.4846,
         -0.3260, -0.4666, -0.5843,  0.3834]], device='cuda:0')
obs :  tensor([[-0.1341, -0.5763,  0.4862, -0.0357,  0.0068, -0.9993,  1.0000,  0.0000,
          0.0000, -0.0866, -0.0994, -0.0781,  0.1559, -0.5855, -0.0127,  0.0936,
         -0.0238,  0.1360,  0.0723, -0.7330,  0.2830, -0.4354, -0.0955,  0.1708,
         -0.1758,  0.3675,  0.3799,  0.7654,  0.1053,  0.4085, -0.1619, -0.0210,
          0.3547,  0.1443,  1.2860, -1.1694, -0.3019, -0.7479,  0.3948, -0.1507,
          0.4846, -0.3260, -0.4666, -0.5843,  0.3834]], device='cuda:0')
torques: [ -29.53326261  200.          200.         -200.          200.
   49.93601733  -20.05690899  200.         -117.55966012  -66.95428114
  200.          -28.55968054]
データ収集:

In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[ 0.8255, -0.9202, -1.1438,  0.8056,  0.0791, -0.1052, -0.9192, -1.4572,
         -0.5552, -1.0898,  0.3781, -0.6812]], device='cuda:0')
Scaled actions :  tensor([[ 0.8255, -0.9202, -1.1438,  0.8056,  0.0791, -0.1052, -0.9192, -1.4572,
         -0.5552, -1.0898,  0.3781, -0.6812]], device='cuda:0')
obs :  tensor([[-0.7599,  0.0144,  0.6151, -0.0457,  0.0267, -0.9986,  1.0000,  0.0000,
          0.0000, -0.1286, -0.0947, -0.0564,  0.1074, -0.6086,  0.1271,  0.1986,
          0.0265,  0.1674,  0.0601, -0.6895,  0.3425, -0.0254,  0.1287,  0.0526,
         -0.2961, -0.3056,  0.7935,  0.3377,  0.3399, -0.0073, -0.0453,  0.2185,
          0.1140,  0.8255, -0.9202, -1.1438,  0.8056,  0.0791, -0.1052, -0.9192,
         -1.4572, -0.5552, -1.0898,  0.3781, -0.6812]], device='cuda:0')
torques: [ 200.          200.         -200.         -200.           91.76481258
 -183.85120711 -200.          200.         -200.         -200.
   -8.47200893  -82.13292649]
データ収集:

In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[ 0.2024, -0.4854,  0.2526,  0.2125,  0.1639,  0.0037, -0.1912, -0.7539,
          1.3019, -0.2015, -0.6341, -0.0796]], device='cuda:0')
Scaled actions :  tensor([[ 0.2024, -0.4854,  0.2526,  0.2125,  0.1639,  0.0037, -0.1912, -0.7539,
          1.3019, -0.2015, -0.6341, -0.0796]], device='cuda:0')
obs :  tensor([[-0.4006,  0.6291,  0.5127, -0.0307,  0.0501, -0.9983,  1.0000,  0.0000,
          0.0000, -0.0817, -0.0907, -0.0827,  0.0780, -0.5605,  0.1812,  0.2105,
          0.0679,  0.1386,  0.0523, -0.5452,  0.2560,  0.4401, -0.0696, -0.2969,
          0.0246,  0.6909, -0.1629, -0.1669,  0.1045, -0.2325, -0.0552,  1.1499,
         -0.8692,  0.2024, -0.4854,  0.2526,  0.2125,  0.1639,  0.0037, -0.1912,
         -0.7539,  1.3019, -0.2015, -0.6341, -0.0796]], device='cuda:0')
torques: [ 200. -200. -200.  200.  200. -200. -200. -200. -200. -200.  200. -200.]
データ収集: step 8


In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[-0.6887,  0.4883,  0.1061,  0.7830, -1.4625,  0.1664,  0.2869, -0.4513,
          1.2910, -1.1277, -0.5686, -0.0109]], device='cuda:0')
Scaled actions :  tensor([[-0.6887,  0.4883,  0.1061,  0.7830, -1.4625,  0.1664,  0.2869, -0.4513,
          1.2910, -1.1277, -0.5686, -0.0109]], device='cuda:0')
obs :  tensor([[ 0.0817, -0.0410,  0.8041, -0.0187,  0.0565, -0.9982,  1.0000,  0.0000,
          0.0000,  0.0274, -0.1366, -0.1139,  0.0905, -0.3537,  0.1116,  0.1216,
          0.0717,  0.1267,  0.0364, -0.4295,  0.1058,  0.4597, -0.3390, -0.0356,
          0.0821,  1.1107, -0.2389, -0.6216, -0.0690,  0.0589, -0.0492,  0.0933,
         -0.4663, -0.6887,  0.4883,  0.1061,  0.7830, -1.4625,  0.1664,  0.2869,
         -0.4513,  1.2910, -1.1277, -0.5686, -0.0109]], device='cuda:0')
torques: [-113.66193909 -200.          200.          168.33895505  -61.48595795
   29.51062307  -49.82829418 -200.          200.         -200.
 -200.          115.53560009]
データ収集:

In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-0.8270,  0.7160, -1.2101,  0.3072, -1.7408,  0.5695,  1.3807, -0.3594,
         -0.9683, -1.5216, -0.6926, -0.3571]], device='cuda:0')
Scaled actions :  tensor([[-0.8270,  0.7160, -1.2101,  0.3072, -1.7408,  0.5695,  1.3807, -0.3594,
         -0.9683, -1.5216, -0.6926, -0.3571]], device='cuda:0')
obs :  tensor([[ 0.0211, -0.7609,  0.7666, -0.0340,  0.0552, -0.9979,  1.0000,  0.0000,
          0.0000,  0.0686, -0.1865, -0.0970,  0.1190, -0.2396,  0.1127,  0.0354,
          0.0616,  0.1724,  0.0171, -0.4703, -0.0082,  0.0051, -0.1717,  0.1790,
          0.1927,  0.1274,  0.0974, -0.2852, -0.0329,  0.3505, -0.0973, -0.2210,
         -0.3772, -0.8270,  0.7160, -1.2101,  0.3072, -1.7408,  0.5695,  1.3807,
         -0.3594, -0.9683, -1.5216, -0.6926, -0.3571]], device='cuda:0')
torques: [-200.          200.          200.          200.         -200.
   19.54880024  200.         -200.          200.         -200.
   29.50459769  200.        ]
データ収集: step 10

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[ 0.1171, -0.1236, -1.2119,  0.2188, -0.4839,  0.5365,  0.6601, -0.7901,
         -0.7608, -1.2600, -0.0645, -0.2924]], device='cuda:0')
Scaled actions :  tensor([[ 0.1171, -0.1236, -1.2119,  0.2188, -0.4839,  0.5365,  0.6601, -0.7901,
         -0.7608, -1.2600, -0.0645, -0.2924]], device='cuda:0')
obs :  tensor([[-0.1091, -0.2062,  0.6676, -0.0511,  0.0591, -0.9969,  1.0000,  0.0000,
          0.0000,  0.0079, -0.2047, -0.1020,  0.1773, -0.3193,  0.2261,  0.0187,
          0.0690,  0.2141,  0.0233, -0.5387, -0.1005, -0.5612, -0.0306, -0.1865,
          0.3306, -0.8275,  0.7426,  0.0974,  0.0727,  0.0995,  0.0644, -0.4834,
         -0.6088,  0.1171, -0.1236, -1.2119,  0.2188, -0.4839,  0.5365,  0.6601,
         -0.7901, -0.7608, -1.2600, -0.0645, -0.2924]], device='cuda:0')
torques: [-200.          200.         -200.          -45.58895754 -200.
  -95.28166372  200.         -200.         -200.         -200.
 -200.         -200.        ]
データ収集: step 1

In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[ 0.5723,  0.0756,  0.2865, -0.5851,  0.2474, -0.2475,  0.0238, -0.1848,
          0.5374, -1.3571,  0.6462,  0.2007]], device='cuda:0')
Scaled actions :  tensor([[ 0.5723,  0.0756,  0.2865, -0.5851,  0.2474, -0.2475,  0.0238, -0.1848,
          0.5374, -1.3571,  0.6462,  0.2007]], device='cuda:0')
obs :  tensor([[-0.1604,  0.4405, -0.0967, -0.0465,  0.0649, -0.9968,  1.0000,  0.0000,
          0.0000, -0.0563, -0.2057, -0.1570,  0.2384, -0.4625,  0.4698,  0.0909,
          0.0889,  0.1999,  0.0386, -0.5255, -0.2475, -0.1339,  0.0202, -0.3405,
          0.2480, -0.3867,  1.3259,  0.5778,  0.1041, -0.2201,  0.0833,  0.5046,
         -0.7565,  0.5723,  0.0756,  0.2865, -0.5851,  0.2474, -0.2475,  0.0238,
         -0.1848,  0.5374, -1.3571,  0.6462,  0.2007]], device='cuda:0')
torques: [ 200.          160.9163569  -200.         -200.          200.
 -200.          200.         -200.         -200.         -200.
  200.         -149.00683434]
データ収集: step 1

In [37]:
# 既存のforループを置き換え
num_steps = 10
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=0.829, Scaled action max=0.829
Step 1/10, Total steps: 142
steps: 142
actions : tensor([[-0.6583, -0.7762, -0.5239,  0.3898, -0.3157,  0.2155, -0.2149, -1.7650,
          0.1029, -1.3063,  0.8288, -1.0272]], device='cuda:0')
target_dof_pos: tensor([[ 0.0835, -1.1348, -1.2592,  1.2923, -0.3872, -0.0610, -1.3185, -2.0435,
         -0.2174,  0.0444, -0.2278, -1.1435]], device='cuda:0')
Step 1: Original action max=1.392, Scaled action max=1.392
Step 2: Original action max=1.141, Scaled action max=1.141
データ収集完了: 10 steps collected with action_scale=1.0


In [38]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [39]:
env.sim.stop()

In [44]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.9.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-fractal-kp2000kd50_ckpt100_scale1.0_rotorInertia0.9.csv
データ形状: (192, 58)
